# Randomized-encoding circuit QPINN data generator

This notebook extends the one-dimensional DSQE/QPINN experiment in `DSQE_QPINN.ipynb` to a two-dimensional PDE with hyperoctahedral symmetry.  Unlike a deterministic average over all group elements, the averaged model is evaluated by randomized encoding: each shot samples one input transformation and runs the same baseline circuit.

The PDE is the screened Poisson equation on $\Omega=[-1,1]^2$,

$$
(-\Delta+\lambda)u(x)=q(x), \qquad u|_{\partial\Omega}=0 .
$$

We use the same smooth $B_2$-invariant source as before.  Let

$$
\phi_m(t)=\cos\left(\frac{m\pi t}{2}\right).
$$

The source is

$$
q(x_1,x_2)=\phi_1(x_1)\phi_1(x_2)
+0.3\left[\phi_1(x_1)\phi_3(x_2)+\phi_3(x_1)\phi_1(x_2)\right],
$$

and the analytical solution is

$$
u_\star(x_1,x_2)=
\frac{\phi_1(x_1)\phi_1(x_2)}{\lambda+\pi^2/2}
+\frac{0.3\left[\phi_1(x_1)\phi_3(x_2)+\phi_3(x_1)\phi_1(x_2)\right]}{\lambda+5\pi^2/2}.
$$

The solution is invariant under sign flips and coordinate exchange.  Therefore the relevant group is the hyperoctahedral group $B_2=\{\pm1\}^2\rtimes S_2$.

The circuit QFM is a data re-uploading model.  Each upload block applies a trainable StronglyEntanglingLayer block followed by exponential Pauli encoding,

$$
R_X(\beta_j x_j)=\exp[-i\beta_j x_jX_j/2],
\qquad \beta_j=3^j .
$$

All spatial derivatives used in the QPINN residual are estimated by gate-level parameter-shift rules.  The notebook therefore avoids `torch.autograd` for the PDE residual and for the training gradient.

This split project keeps training and plotting separate.  Run this notebook first.  It writes the plot-ready arrays to `data/qpinn_b2_randomized_encoding_data.npz`; it does not create manuscript figures.


In [ ]:
from __future__ import annotations

from collections import defaultdict
from copy import deepcopy
from itertools import combinations
from pathlib import Path
import os
import sys
import tempfile
import time

import numpy as np
import torch

_MPL_CACHE = Path(tempfile.gettempdir()) / "matplotlib-cache"
_MPL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(_MPL_CACHE))
_XDG_CACHE = Path(tempfile.gettempdir()) / "xdg-cache"
_XDG_CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("XDG_CACHE_HOME", str(_XDG_CACHE))

IS_NOTEBOOK = "ipykernel" in sys.modules
import matplotlib
if not IS_NOTEBOOK:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt


def show_or_close_current_figure() -> None:
    if IS_NOTEBOOK:
        plt.show()
    else:
        plt.close()


def _device_works(candidate: torch.device) -> bool:
    try:
        real = torch.float32
        comp = torch.complex64
        theta = torch.tensor([0.1, 0.2], dtype=real, device=candidate)
        gate = torch.complex(torch.cos(theta), torch.sin(theta)).to(comp)
        state = torch.ones(2, dtype=comp, device=candidate)
        test = (gate * state).abs().sum()
        return bool(torch.isfinite(test).item())
    except Exception as exc:
        print(f"Skipping {candidate.type}: complex statevector check failed ({exc}).")
        return False


def select_accelerated_device() -> torch.device:
    candidates = []
    if torch.cuda.is_available():
        candidates.append(torch.device("cuda"))
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        candidates.append(torch.device("mps"))
    candidates.append(torch.device("cpu"))
    for candidate in candidates:
        if _device_works(candidate):
            return candidate
    return torch.device("cpu")


DEVICE = select_accelerated_device()
REAL_DTYPE = torch.float32
COMPLEX_DTYPE = torch.complex64

SEED = 7
torch.manual_seed(SEED)
np.random.seed(SEED)



def locate_project_root() -> Path:
    cwd = Path.cwd().resolve()
    markers = (
        "01_train_and_save_data.ipynb",
        "02_plot_saved_data.ipynb",
        "03_train_randomized_encoding_save_data.ipynb",
        "04_plot_randomized_encoding_saved_data.ipynb",
        "requirements.txt",
    )
    for candidate in [cwd, *cwd.parents]:
        if any((candidate / marker).exists() for marker in markers):
            return candidate
    return cwd


PROJECT_ROOT = locate_project_root()
DATA_DIR = PROJECT_ROOT / "data"
FIGURE_DIR = PROJECT_ROOT / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root={PROJECT_ROOT}")
print(f"Using device={DEVICE}, real dtype={REAL_DTYPE}, complex dtype={COMPLEX_DTYPE}")


## Randomized encoding and parameter-shift residual

For a finite input-transformation group $G$, the averaged predictor is

$$
(Af_\theta)(x)=\mathbb E_{g\sim\mathrm{Unif}(G)}[f_\theta(gx)].
$$

The implementation below never enumerates all elements of $G$ during training.  Each circuit shot samples one transformation $g$ and evaluates the baseline circuit on $gx$.  This matches the randomized-encoding estimator in the manuscript.

For a re-uploading circuit, a coordinate $x_j$ may enter several data gates.  Therefore the spatial derivative cannot be obtained by shifting the whole coordinate once.  If $\varphi_r=\beta_r x_j$ denotes the angle of the $r$-th encoding gate depending on $x_j$, then

$$
\frac{\partial^2 f}{\partial x_j^2}
=
\sum_r \beta_r^2\frac{\partial^2 f}{\partial\varphi_r^2}
+2\sum_{r<s}\beta_r\beta_s
\frac{\partial^2 f}{\partial\varphi_r\partial\varphi_s}.
$$

For Pauli rotations,

$$
\frac{\partial^2 f}{\partial\varphi_r^2}
=\frac{1}{2}\left[f(\varphi_r+\pi)-f(\varphi_r)\right],
$$

and the mixed term is

$$
\frac{\partial^2 f}{\partial\varphi_r\partial\varphi_s}
=\frac{1}{4}\left[
f_{++}-f_{+-}-f_{-+}+f_{--}
\right],
$$

where $f_{\pm\pm}$ shifts the two selected encoding gates by $\pm\pi/2$.  Because sign flips and coordinate permutations are orthogonal transformations, the Laplacian commutes with the $B_2$ average.  Thus randomized encoding gives an unbiased estimator for both $Af_\theta$ and $\Delta Af_\theta$.


In [ ]:
DIM = 2
N_QUBITS = 2
N_UPLOAD_LAYERS = 2
STRONG_LAYERS_PER_BLOCK = 1

EXP_BASE = 3.0
ENCODING_SCALE = 1.0
DIFF_GENERATOR_PER_LAYER = False

LAMBDA = 1.0
SOURCE_MODE_COEFFICIENTS = {
    (1, 1): 1.0,
    (1, 3): 0.3,
    (3, 1): 0.3,
}

# These defaults are intentionally light because manual PSR gradients scale
# with the number of trainable gates. Increase shots/steps for final figures.
TRAINING_STEPS = 50
INTERIOR_BATCH = 2
BOUNDARY_BATCH = 2
N_SHOTS_TRAIN = 8
N_SHOTS_EVAL = 128
USE_MEASUREMENT_SHOTS = False

LR_THETA = 0.03
LR_AFFINE = 0.05
BOUNDARY_WEIGHT = 10.0
INTERIOR_MARGIN = 0.94
GRAD_CLIP = 5.0

EVAL_GRID_N = 21

PARAM_SHIFT = np.pi / 2

assert DIM == 2, "This notebook implements the B_2 screened-Poisson example."
assert N_QUBITS >= DIM, "The circuit uses one encoding wire per input coordinate."


In [ ]:
_CNOT_GATHER_CACHE: dict[tuple[int, int, int, str], torch.Tensor] = {}


def as_1d(theta: torch.Tensor) -> torch.Tensor:
    return theta.reshape(1) if theta.ndim == 0 else theta


def complex_pair(real: torch.Tensor, imag: torch.Tensor) -> torch.Tensor:
    return torch.complex(real, imag).to(COMPLEX_DTYPE)


def rx_matrix(theta: torch.Tensor) -> torch.Tensor:
    theta = as_1d(theta)
    half = 0.5 * theta
    c = torch.cos(half)
    s = torch.sin(half)
    z = torch.zeros_like(c)
    row0 = torch.stack([complex_pair(c, z), complex_pair(z, -s)], dim=-1)
    row1 = torch.stack([complex_pair(z, -s), complex_pair(c, z)], dim=-1)
    return torch.stack([row0, row1], dim=-2)


def ry_matrix(theta: torch.Tensor) -> torch.Tensor:
    theta = as_1d(theta)
    half = 0.5 * theta
    c = torch.cos(half)
    s = torch.sin(half)
    z = torch.zeros_like(c)
    row0 = torch.stack([complex_pair(c, z), complex_pair(-s, z)], dim=-1)
    row1 = torch.stack([complex_pair(s, z), complex_pair(c, z)], dim=-1)
    return torch.stack([row0, row1], dim=-2)


def rz_matrix(theta: torch.Tensor) -> torch.Tensor:
    theta = as_1d(theta)
    half = 0.5 * theta
    c = torch.cos(half)
    s = torch.sin(half)
    z = torch.zeros_like(c)
    row0 = torch.stack([complex_pair(c, -s), complex_pair(z, z)], dim=-1)
    row1 = torch.stack([complex_pair(z, z), complex_pair(c, s)], dim=-1)
    return torch.stack([row0, row1], dim=-2)


def rot_matrix(phi: torch.Tensor, theta: torch.Tensor, omega: torch.Tensor) -> torch.Tensor:
    return rz_matrix(omega) @ ry_matrix(theta) @ rz_matrix(phi)


def apply_single_qubit_gate(state: torch.Tensor, gate: torch.Tensor, wire: int) -> torch.Tensor:
    batch = state.shape[0]
    if gate.shape[0] == 1 and batch != 1:
        gate = gate.expand(batch, -1, -1)
    tensor = state.reshape(batch, *([2] * N_QUBITS))
    axis = wire + 1
    tensor = tensor.movedim(axis, -1)
    updated = torch.einsum("bij,b...j->b...i", gate, tensor)
    updated = updated.movedim(-1, axis)
    return updated.reshape(batch, 2**N_QUBITS)


def cnot_gather_index(n_qubits: int, control: int, target: int, device: torch.device) -> torch.Tensor:
    key = (n_qubits, control, target, device.type)
    if key in _CNOT_GATHER_CACHE:
        return _CNOT_GATHER_CACHE[key]

    gather = [0] * (2**n_qubits)
    for input_index in range(2**n_qubits):
        bits = [(input_index >> (n_qubits - 1 - q)) & 1 for q in range(n_qubits)]
        output_bits = bits[:]
        if bits[control] == 1:
            output_bits[target] ^= 1
        output_index = 0
        for bit in output_bits:
            output_index = (output_index << 1) | bit
        gather[output_index] = input_index

    tensor = torch.tensor(gather, dtype=torch.long, device=device)
    _CNOT_GATHER_CACHE[key] = tensor
    return tensor


def apply_cnot(state: torch.Tensor, control: int, target: int) -> torch.Tensor:
    gather = cnot_gather_index(N_QUBITS, control, target, state.device)
    return state[:, gather]


def apply_strongly_entangling_block(state: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
    for sublayer in range(weights.shape[0]):
        for wire in range(N_QUBITS):
            phi, theta, omega = weights[sublayer, wire]
            state = apply_single_qubit_gate(state, rot_matrix(phi, theta, omega), wire)
        if N_QUBITS > 1:
            cnot_range = (sublayer % (N_QUBITS - 1)) + 1
            for control in range(N_QUBITS):
                target = (control + cnot_range) % N_QUBITS
                state = apply_cnot(state, control, target)
    return state


def initial_state(batch: int) -> torch.Tensor:
    state = torch.zeros(batch, 2**N_QUBITS, dtype=COMPLEX_DTYPE, device=DEVICE)
    state[:, 0] = torch.ones(batch, dtype=COMPLEX_DTYPE, device=DEVICE)
    return state


def z0_values() -> torch.Tensor:
    values = []
    for basis_index in range(2**N_QUBITS):
        bit0 = (basis_index >> (N_QUBITS - 1)) & 1
        values.append(1.0 if bit0 == 0 else -1.0)
    return torch.tensor(values, dtype=REAL_DTYPE, device=DEVICE)


Z0_VALUES = z0_values()


In [ ]:
def encoding_beta(upload: int, wire: int) -> float:
    exponent = upload * N_QUBITS + wire if DIFF_GENERATOR_PER_LAYER else wire
    return EXP_BASE ** exponent


def data_gate_key(upload: int, wire: int) -> tuple[int, int]:
    return (upload, wire)


def shifted_value(input_gate_shifts: dict[tuple[int, int], float] | None, upload: int, wire: int) -> float:
    if input_gate_shifts is None:
        return 0.0
    return float(input_gate_shifts.get(data_gate_key(upload, wire), 0.0))


def circuit_z0_expectation(
    transformed_x: torch.Tensor,
    theta: torch.Tensor,
    input_gate_shifts: dict[tuple[int, int], float] | None = None,
) -> torch.Tensor:
    batch = transformed_x.shape[0]
    state = initial_state(batch)

    for upload in range(N_UPLOAD_LAYERS):
        state = apply_strongly_entangling_block(state, theta[upload])
        for wire in range(N_QUBITS):
            coordinate = wire % DIM
            beta = encoding_beta(upload, wire)
            angle = ENCODING_SCALE * beta * transformed_x[:, coordinate]
            angle = angle + shifted_value(input_gate_shifts, upload, wire)
            state = apply_single_qubit_gate(state, rx_matrix(angle), wire)

    state = apply_strongly_entangling_block(state, theta[-1])
    probabilities = (state.abs() ** 2).real
    return probabilities @ Z0_VALUES


def sample_group_transformed_inputs(points: torch.Tensor, group: str, shots: int) -> torch.Tensor:
    points = points.to(device=DEVICE, dtype=REAL_DTYPE)
    expanded = points[:, None, :].expand(points.shape[0], shots, DIM).clone()

    if group == "none":
        transformed = expanded
    elif group == "signflip":
        signs = torch.where(
            torch.rand(expanded.shape, device=DEVICE) < 0.5,
            -torch.ones_like(expanded),
            torch.ones_like(expanded),
        )
        transformed = expanded * signs
    elif group == "hyperoctahedral":
        swap = torch.rand(expanded.shape[:2], device=DEVICE) < 0.5
        swapped = expanded[..., [1, 0]]
        transformed = torch.where(swap[..., None], swapped, expanded)
        signs = torch.where(
            torch.rand(transformed.shape, device=DEVICE) < 0.5,
            -torch.ones_like(transformed),
            torch.ones_like(transformed),
        )
        transformed = transformed * signs
    else:
        raise ValueError(f"Unknown group: {group}")

    return transformed.reshape(-1, DIM)


def estimate_raw_expectation(
    points: torch.Tensor,
    theta: torch.Tensor,
    group: str,
    shots: int,
    input_gate_shifts: dict[tuple[int, int], float] | None = None,
    sample_measurements: bool | None = None,
) -> torch.Tensor:
    if sample_measurements is None:
        sample_measurements = USE_MEASUREMENT_SHOTS
    transformed = sample_group_transformed_inputs(points, group=group, shots=shots)
    z_exp = circuit_z0_expectation(transformed, theta, input_gate_shifts=input_gate_shifts).clamp(-1.0, 1.0)
    if sample_measurements:
        prob_plus = ((1.0 + z_exp) / 2.0).clamp(0.0, 1.0)
        samples = torch.where(torch.rand_like(prob_plus) < prob_plus, 1.0, -1.0)
        values = samples
    else:
        values = z_exp
    return values.reshape(points.shape[0], shots).mean(dim=1)


def coordinate_data_gates(coordinate: int) -> list[tuple[int, int]]:
    return [
        data_gate_key(upload, wire)
        for upload in range(N_UPLOAD_LAYERS)
        for wire in range(N_QUBITS)
        if wire % DIM == coordinate
    ]


def merge_shifts(*items: tuple[tuple[int, int], float]) -> dict[tuple[int, int], float]:
    merged: dict[tuple[int, int], float] = defaultdict(float)
    for gate, shift in items:
        merged[gate] += float(shift)
    return dict(merged)


def estimate_raw_u_and_laplacian(
    points: torch.Tensor,
    theta: torch.Tensor,
    group: str,
    shots: int,
    sample_measurements: bool | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    base = estimate_raw_expectation(points, theta, group, shots, sample_measurements=sample_measurements)
    laplacian = torch.zeros_like(base)

    for coordinate in range(DIM):
        gates = coordinate_data_gates(coordinate)
        second = torch.zeros_like(base)

        for gate in gates:
            upload, wire = gate
            beta = encoding_beta(upload, wire)
            shifted = estimate_raw_expectation(
                points,
                theta,
                group,
                shots,
                input_gate_shifts={gate: np.pi},
                sample_measurements=sample_measurements,
            )
            second = second + 0.5 * (beta**2) * (shifted - base)

        for gate_a, gate_b in combinations(gates, 2):
            ua, wa = gate_a
            ub, wb = gate_b
            beta_a = encoding_beta(ua, wa)
            beta_b = encoding_beta(ub, wb)
            pp = estimate_raw_expectation(points, theta, group, shots, merge_shifts((gate_a, PARAM_SHIFT), (gate_b, PARAM_SHIFT)), sample_measurements)
            pm = estimate_raw_expectation(points, theta, group, shots, merge_shifts((gate_a, PARAM_SHIFT), (gate_b, -PARAM_SHIFT)), sample_measurements)
            mp = estimate_raw_expectation(points, theta, group, shots, merge_shifts((gate_a, -PARAM_SHIFT), (gate_b, PARAM_SHIFT)), sample_measurements)
            mm = estimate_raw_expectation(points, theta, group, shots, merge_shifts((gate_a, -PARAM_SHIFT), (gate_b, -PARAM_SHIFT)), sample_measurements)
            second = second + 0.5 * beta_a * beta_b * (pp - pm - mp + mm)

        laplacian = laplacian + second

    return base, laplacian


In [ ]:
def phi_mode(x: torch.Tensor, mode: int) -> torch.Tensor:
    return torch.cos(0.5 * mode * torch.pi * x)


def source_term(x: torch.Tensor) -> torch.Tensor:
    value = torch.zeros(x.shape[0], dtype=x.dtype, device=x.device)
    for (m1, m2), coefficient in SOURCE_MODE_COEFFICIENTS.items():
        value = value + coefficient * phi_mode(x[:, 0], m1) * phi_mode(x[:, 1], m2)
    return value


def analytical_solution(x: torch.Tensor) -> torch.Tensor:
    value = torch.zeros(x.shape[0], dtype=x.dtype, device=x.device)
    for (m1, m2), coefficient in SOURCE_MODE_COEFFICIENTS.items():
        eigenvalue = (0.5 * torch.pi * m1) ** 2 + (0.5 * torch.pi * m2) ** 2
        value = value + coefficient * phi_mode(x[:, 0], m1) * phi_mode(x[:, 1], m2) / (LAMBDA + eigenvalue)
    return value


def sample_interior(batch: int) -> torch.Tensor:
    return (2.0 * torch.rand(batch, DIM, dtype=REAL_DTYPE, device=DEVICE) - 1.0) * INTERIOR_MARGIN


def sample_boundary(batch: int) -> torch.Tensor:
    x = 2.0 * torch.rand(batch, DIM, dtype=REAL_DTYPE, device=DEVICE) - 1.0
    sides = torch.randint(0, 2 * DIM, (batch,), device=DEVICE)
    for dim in range(DIM):
        left = sides == 2 * dim
        right = sides == 2 * dim + 1
        x[left, dim] = -1.0
        x[right, dim] = 1.0
    return x


def make_initial_params(init_scale: float = 0.2) -> dict[str, torch.Tensor]:
    theta_shape = (N_UPLOAD_LAYERS + 1, STRONG_LAYERS_PER_BLOCK, N_QUBITS, 3)
    return {
        "theta": init_scale * torch.randn(theta_shape, dtype=REAL_DTYPE, device=DEVICE),
        "scale": torch.tensor(0.5, dtype=REAL_DTYPE, device=DEVICE),
        "bias": torch.tensor(0.0, dtype=REAL_DTYPE, device=DEVICE),
    }


def clone_params(params: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    return {key: value.clone() for key, value in params.items()}


def shift_theta(theta: torch.Tensor, flat_index: int, shift: float) -> torch.Tensor:
    shifted = theta.clone()
    shifted.reshape(-1)[flat_index] += float(shift)
    return shifted


def model_output_from_raw(raw_u: torch.Tensor, params: dict[str, torch.Tensor]) -> torch.Tensor:
    return params["scale"] * raw_u + params["bias"]


def model_laplacian_from_raw(raw_laplacian: torch.Tensor, params: dict[str, torch.Tensor]) -> torch.Tensor:
    return params["scale"] * raw_laplacian


def loss_terms(
    params: dict[str, torch.Tensor],
    group: str,
    interior_x: torch.Tensor,
    boundary_x: torch.Tensor,
    shots: int,
) -> dict[str, torch.Tensor]:
    raw_u, raw_lap = estimate_raw_u_and_laplacian(interior_x, params["theta"], group, shots)
    u = model_output_from_raw(raw_u, params)
    lap = model_laplacian_from_raw(raw_lap, params)
    residual = -lap + LAMBDA * u - source_term(interior_x)

    raw_boundary = estimate_raw_expectation(boundary_x, params["theta"], group, shots)
    boundary_u = model_output_from_raw(raw_boundary, params)

    residual_loss = torch.mean(residual**2)
    boundary_loss = torch.mean(boundary_u**2)
    total_loss = residual_loss + BOUNDARY_WEIGHT * boundary_loss

    return {
        "loss": total_loss,
        "residual_loss": residual_loss,
        "boundary_loss": boundary_loss,
        "residual": residual,
        "boundary_u": boundary_u,
        "raw_u": raw_u,
        "raw_lap": raw_lap,
        "raw_boundary": raw_boundary,
    }


## Manual PSR training gradient

The residual is

$$
r_\theta(x)=-(\Delta u_\theta)(x)+\lambda u_\theta(x)-q(x).
$$

For the empirical loss

$$
\mathcal L(\theta)=
\frac{1}{N_\Omega}\sum_i r_\theta(x_i)^2
+\mu\frac{1}{N_{\partial\Omega}}\sum_i u_\theta(y_i)^2,
$$

the circuit-parameter derivative is evaluated by nested parameter shift:

$$
\partial_{\theta_k}r_\theta
=
-\Delta(\partial_{\theta_k}u_\theta)+\lambda\partial_{\theta_k}u_\theta,
\qquad
\partial_{\theta_k}u_\theta
=\frac{u_{\theta_k+\pi/2}-u_{\theta_k-\pi/2}}{2}.
$$

The same shift is applied inside the Laplacian estimator, so no automatic differentiation is used for the PDE derivatives or training gradient.  The only trainable non-circuit parameters are the scalar affine readout $u=a\langle Z_0\rangle+b$, whose gradients are analytic.


In [ ]:
def manual_loss_and_gradient(
    params: dict[str, torch.Tensor],
    group: str,
    interior_x: torch.Tensor,
    boundary_x: torch.Tensor,
    shots: int,
) -> tuple[float, dict[str, torch.Tensor], dict[str, float]]:
    terms = loss_terms(params, group, interior_x, boundary_x, shots)
    residual = terms["residual"]
    boundary_u = terms["boundary_u"]
    raw_u = terms["raw_u"]
    raw_lap = terms["raw_lap"]
    raw_boundary = terms["raw_boundary"]

    theta_grad = torch.zeros_like(params["theta"])
    flat_grad = theta_grad.reshape(-1)

    for flat_index in range(params["theta"].numel()):
        theta_plus = shift_theta(params["theta"], flat_index, PARAM_SHIFT)
        theta_minus = shift_theta(params["theta"], flat_index, -PARAM_SHIFT)

        plus_u, plus_lap = estimate_raw_u_and_laplacian(interior_x, theta_plus, group, shots)
        minus_u, minus_lap = estimate_raw_u_and_laplacian(interior_x, theta_minus, group, shots)
        d_raw_u = 0.5 * (plus_u - minus_u)
        d_raw_lap = 0.5 * (plus_lap - minus_lap)
        d_residual = params["scale"] * (-d_raw_lap + LAMBDA * d_raw_u)

        plus_boundary = estimate_raw_expectation(boundary_x, theta_plus, group, shots)
        minus_boundary = estimate_raw_expectation(boundary_x, theta_minus, group, shots)
        d_boundary = params["scale"] * 0.5 * (plus_boundary - minus_boundary)

        flat_grad[flat_index] = 2.0 * torch.mean(residual * d_residual) + 2.0 * BOUNDARY_WEIGHT * torch.mean(boundary_u * d_boundary)

    d_residual_d_scale = -raw_lap + LAMBDA * raw_u
    d_boundary_d_scale = raw_boundary
    scale_grad = 2.0 * torch.mean(residual * d_residual_d_scale) + 2.0 * BOUNDARY_WEIGHT * torch.mean(boundary_u * d_boundary_d_scale)
    bias_grad = 2.0 * torch.mean(residual * LAMBDA) + 2.0 * BOUNDARY_WEIGHT * torch.mean(boundary_u)

    theta_norm = torch.linalg.norm(theta_grad)
    if theta_norm > GRAD_CLIP:
        theta_grad = theta_grad * (GRAD_CLIP / (theta_norm + 1.0e-12))
    scale_grad = torch.clamp(scale_grad, -GRAD_CLIP, GRAD_CLIP)
    bias_grad = torch.clamp(bias_grad, -GRAD_CLIP, GRAD_CLIP)

    grads = {
        "theta": theta_grad,
        "scale": scale_grad,
        "bias": bias_grad,
    }
    metrics = {
        "loss": float(terms["loss"].detach().cpu()),
        "residual": float(terms["residual_loss"].detach().cpu()),
        "boundary": float(terms["boundary_loss"].detach().cpu()),
        "theta_grad_norm": float(torch.linalg.norm(theta_grad).detach().cpu()),
    }
    return metrics["loss"], grads, metrics


def apply_manual_update(params: dict[str, torch.Tensor], grads: dict[str, torch.Tensor]) -> None:
    params["theta"] = params["theta"] - LR_THETA * grads["theta"]
    params["scale"] = params["scale"] - LR_AFFINE * grads["scale"]
    params["bias"] = params["bias"] - LR_AFFINE * grads["bias"]


MODEL_SPECS = {
    "ordinary QFM": "none",
    "random sign-flip QFM": "signflip",
    "random B2 QFM": "hyperoctahedral",
}


def train_model(name: str, group: str, initial_params: dict[str, torch.Tensor]) -> tuple[dict[str, torch.Tensor], dict[str, list[float]]]:
    params = clone_params(initial_params)
    history = defaultdict(list)
    start = time.time()
    print(f"\nTraining {name} with group='{group}'")

    for step in range(1, TRAINING_STEPS + 1):
        interior_x = sample_interior(INTERIOR_BATCH)
        boundary_x = sample_boundary(BOUNDARY_BATCH)
        _, grads, metrics = manual_loss_and_gradient(params, group, interior_x, boundary_x, N_SHOTS_TRAIN)
        apply_manual_update(params, grads)

        for key, value in metrics.items():
            history[key].append(value)

        if step == 1 or step % max(1, TRAINING_STEPS // 4) == 0:
            print(
                f"{name:22s} step {step:3d}/{TRAINING_STEPS}: "
                f"loss={metrics['loss']:.3e}, residual={metrics['residual']:.3e}, "
                f"boundary={metrics['boundary']:.3e}, |grad|={metrics['theta_grad_norm']:.3e}"
            )

    print(f"{name:22s} finished in {time.time() - start:.1f} s; scale={float(params['scale']):.3f}, bias={float(params['bias']):.3f}")
    return params, dict(history)


## Train randomized-encoding QPINNs

This cell trains the ordinary QFM, randomized sign-flip QFM, and randomized $B_2$ QFM from the same initialization.


In [ ]:
torch.manual_seed(SEED)
initial_params = make_initial_params()

trained_params = {}
histories = {}
for model_name, group in MODEL_SPECS.items():
    params, history = train_model(model_name, group, initial_params)
    trained_params[model_name] = params
    histories[model_name] = history


## Evaluate representative grids

The trained models are evaluated on a fixed square grid.  These arrays are saved directly so the plotting notebook does not need to rebuild or rerun the circuits.


In [ ]:
def evaluation_grid(n: int = EVAL_GRID_N) -> tuple[torch.Tensor, np.ndarray, np.ndarray]:
    axis = torch.linspace(-1.0, 1.0, n, dtype=REAL_DTYPE, device=DEVICE)
    x1, x2 = torch.meshgrid(axis, axis, indexing="ij")
    points = torch.stack([x1.reshape(-1), x2.reshape(-1)], dim=1)
    return points, x1.detach().cpu().numpy(), x2.detach().cpu().numpy()


def estimate_model_output(points: torch.Tensor, params: dict[str, torch.Tensor], group: str, shots: int) -> torch.Tensor:
    raw = estimate_raw_expectation(points, params["theta"], group, shots, sample_measurements=False)
    return model_output_from_raw(raw, params)


def relative_l2_error(pred: torch.Tensor, target: torch.Tensor) -> float:
    return float(torch.linalg.norm(pred - target).cpu() / torch.linalg.norm(target).cpu())


def symmetry_defects(params: dict[str, torch.Tensor], group: str, points: torch.Tensor, shots: int) -> dict[str, float]:
    base = estimate_model_output(points, params, group, shots)
    sign_flip = estimate_model_output(points * torch.tensor([-1.0, 1.0], dtype=REAL_DTYPE, device=DEVICE), params, group, shots)
    permutation = estimate_model_output(points[:, [1, 0]], params, group, shots)
    return {
        "sign_flip_mse": float(torch.mean((base - sign_flip) ** 2).cpu()),
        "permutation_mse": float(torch.mean((base - permutation) ** 2).cpu()),
    }


grid_points, grid_x1, grid_x2 = evaluation_grid()
target = analytical_solution(grid_points).detach()
target_grid = target.cpu().numpy().reshape(EVAL_GRID_N, EVAL_GRID_N)

predictions = {}
errors = {}
symmetry = {}

for model_name, params in trained_params.items():
    group = MODEL_SPECS[model_name]
    pred = estimate_model_output(grid_points, params, group, N_SHOTS_EVAL)
    predictions[model_name] = pred.cpu().numpy().reshape(EVAL_GRID_N, EVAL_GRID_N)
    errors[model_name] = relative_l2_error(pred, target)
    symmetry[model_name] = symmetry_defects(params, group, grid_points, N_SHOTS_EVAL)

print("Relative L2 errors")
for model_name, error in errors.items():
    print(f"  {model_name:22s}: {error:.4e}")

print("\nMonte Carlo symmetry defects")
for model_name, defects in symmetry.items():
    print(
        f"  {model_name:22s}: sign-flip MSE={defects['sign_flip_mse']:.3e}, "
        f"permutation MSE={defects['permutation_mse']:.3e}"
    )


## Save plot-ready randomized-encoding data

This cell saves the histories, prediction grids, relative errors, and Monte Carlo symmetry diagnostics needed by the plotting notebook.


In [ ]:
RANDOMIZED_MODEL_SLUGS = {
    "ordinary QFM": "ordinary_qfm",
    "random sign-flip QFM": "random_signflip_qfm",
    "random B2 QFM": "random_b2_qfm",
}

randomized_data_path = DATA_DIR / "qpinn_b2_randomized_encoding_data.npz"
randomized_data = {
    "model_names": np.asarray(list(MODEL_SPECS.keys()), dtype=str),
    "model_groups": np.asarray([MODEL_SPECS[name] for name in MODEL_SPECS], dtype=str),
    "grid_x1": grid_x1,
    "grid_x2": grid_x2,
    "target_grid": target_grid,
    "training_steps": np.asarray(TRAINING_STEPS),
    "interior_batch": np.asarray(INTERIOR_BATCH),
    "boundary_batch": np.asarray(BOUNDARY_BATCH),
    "n_shots_train": np.asarray(N_SHOTS_TRAIN),
    "n_shots_eval": np.asarray(N_SHOTS_EVAL),
    "use_measurement_shots": np.asarray(USE_MEASUREMENT_SHOTS),
    "lambda": np.asarray(LAMBDA),
    "boundary_weight": np.asarray(BOUNDARY_WEIGHT),
    "seed": np.asarray(SEED),
    "eval_grid_n": np.asarray(EVAL_GRID_N),
}

for model_name in MODEL_SPECS:
    slug = RANDOMIZED_MODEL_SLUGS[model_name]
    history = histories[model_name]
    randomized_data[f"{slug}_loss"] = np.asarray(history["loss"], dtype=float)
    randomized_data[f"{slug}_residual"] = np.asarray(history["residual"], dtype=float)
    randomized_data[f"{slug}_boundary"] = np.asarray(history["boundary"], dtype=float)
    randomized_data[f"{slug}_theta_grad_norm"] = np.asarray(history["theta_grad_norm"], dtype=float)
    randomized_data[f"{slug}_prediction_grid"] = predictions[model_name]
    randomized_data[f"{slug}_error"] = np.asarray(errors[model_name])
    randomized_data[f"{slug}_sign_flip_mse"] = np.asarray(symmetry[model_name]["sign_flip_mse"])
    randomized_data[f"{slug}_permutation_mse"] = np.asarray(symmetry[model_name]["permutation_mse"])

np.savez_compressed(randomized_data_path, **randomized_data)
print(f"saved plot-ready randomized-encoding data: {randomized_data_path}")
print("Training data are ready for 04_plot_randomized_encoding_saved_data.ipynb")


## Interpretation

This implementation is intentionally closer to the DSQE estimator than to the deterministic average-operator simulator.  The `random sign-flip QFM` and `random B2 QFM` do not evaluate every element of the group.  Instead, each shot samples one transformed input and runs the unchanged re-uploading circuit.

The reported symmetry defects are finite-shot Monte Carlo diagnostics.  They are not expected to be exactly zero for a finite number of randomized-encoding shots, but they estimate quantities whose expectation vanishes for the corresponding averaged predictor.  Increasing `N_SHOTS_EVAL` should reduce this diagnostic noise.

The training gradient is also obtained without `torch.autograd`: the residual, Laplacian, and circuit-parameter updates are all built from parameter-shift circuit evaluations.  The computational price is that manual PSR training scales with the number of trainable gates, so the default settings are deliberately small and should be increased only for final numerical figures.
